In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych zależności.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Pakiet narzędzi pomocniczych Qiskit addon to zbiór funkcjonalności uzupełniających przepływy pracy obejmujące jeden lub więcej dodatków Qiskit. Na przykład pakiet ten zawiera funkcje do tworzenia hamiltonianów, generowania obwodów ewolucji czasowej Trottera oraz do dzielenia i łączenia obwodów kwantowych.

## Instalacja

Istnieją dwa sposoby instalacji narzędzi pomocniczych Qiskit addon: z PyPI i z kodu źródłowego. Zaleca się instalowanie tych pakietów w [wirtualnym środowisku](https://docs.python.org/3.10/tutorial/venv.html), aby zapewnić separację między zależnościami pakietów.

### Instalacja z PyPI

Najprostszym sposobem instalacji pakietu narzędzi pomocniczych Qiskit addon jest instalacja z PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Instalacja ze źródeł
<details>
<summary>
Kliknij tutaj, aby dowiedzieć się, jak zainstalować ten pakiet ręcznie.
</summary>

Jeśli chcesz wnieść wkład do tego pakietu lub chcesz zainstalować go ręcznie, najpierw sklonuj repozytorium:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

a następnie zainstaluj pakiet przez `pip`. Jeśli planujesz uruchamiać samouczki z repozytorium pakietu, zainstaluj również zależności dla notebooków. Jeśli planujesz pracować nad rozwojem w repozytorium, zainstaluj zależności `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Pierwsze kroki z narzędziami
W pakiecie `qiskit-addon-utils` znajduje się kilka modułów, w tym jeden do generowania problemów do symulacji układów kwantowych, kolorowania grafów w celu wydajniejszego umieszczania bramek w obwodzie kwantowym, oraz dzielenia obwodów, co może pomóc przy [wstecznej propagacji operatorów](/guides/qiskit-addons-obp). Poniższe sekcje podsumowują każdy moduł. Pomocne informacje zawiera również [dokumentacja API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) pakietu.

### Generowanie problemów
Zawartość modułu [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) obejmuje:

- Funkcję [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), która generuje świadomą łączności reprezentację `SparsePauliOp` modelu XYZ typu Isinga:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Funkcję [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), która konstruuje obwód modelujący ewolucję czasową danego operatora.
- Trzy różne obiekty [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) do wyliczania różnych kolejności łańcuchów Pauliego. Jest to przydatne głównie w połączeniu z kolorowaniem grafów i może być używane jako argumenty zarówno w funkcjach `generate_xyz_hamiltonian()`, jak i `generate_time_evolution_circuit()`.

### Kolorowanie grafów
Moduł [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) służy do kolorowania krawędzi w mapie sprzężeń i wykorzystania tego kolorowania do wydajniejszego umieszczania bramek w obwodzie kwantowym. Celem tej pokolorowanej krawędziami mapy sprzężeń jest znalezienie zestawu kolorów krawędzi takiego, że żadne dwie krawędzie tego samego koloru nie współdzielą wspólnego węzła. Dla QPU oznacza to, że bramki wzdłuż krawędzi o tym samym kolorze (połączenia Qubitów) mogą być uruchamiane jednocześnie, a obwód będzie wykonywany szybciej.

Jako krótki przykład możesz użyć funkcji [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges), aby wygenerować kolorowanie krawędzi dla naiwnego obwodu wykonującego `CZGate` wzdłuż każdego połączenia Qubitów. Poniższy fragment kodu używa mapy sprzężeń Backendu `FakeSherbrooke`, tworzy ten naiwny obwód, a następnie używa funkcji `auto_color_edges()` do stworzenia wydajniejszego równoważnego obwodu.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Dzielenie obwodów
Na koniec moduł [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) zawiera funkcje i przebiegi Transpilera do tworzenia „wycinków" obwodów — podobnych do czasu partycji [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) obejmujących wszystkie Qubity. Te wycinki są używane przede wszystkim przy [wstecznej propagacji operatorów](/guides/qiskit-addons-obp-get-started). Cztery główne sposoby podziału obwodu to: według typu bramki, głębokości, kolorowania lub instrukcji [`Barrier`](../api/qiskit/circuit#barrier). Wyniki tych funkcji dzielenia zwracają listę obiektów [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). Podzielone obwody można również ponownie łączyć za pomocą funkcji [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Więcej informacji znajdziesz w [dokumentacji API](../api/qiskit-addon-utils/slicing) modułu.

Poniżej znajduje się kilka przykładów tworzenia tych wycinków na podstawie następującego obwodu:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

W przypadku gdy nie ma jasnego sposobu na wykorzystanie struktury obwodu przy wstecznej propagacji operatorów, możesz podzielić obwód na wycinki o danej głębokości.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Jeśli masz niestandardową strategię dzielenia, możesz zamiast tego umieścić bariery w obwodzie, aby zaznaczyć, gdzie powinien być podzielony, i użyć funkcji [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).